# Food Trust -- Reference Values Build (Epic 3)

Builds `reference_values.db`, a SQLite database of official infant/young-
children nutrition reference figures, transcribed from the NHMRC and WHO
source documents in this folder (`reference_value/`).

Sources used:
- **NHMRC 2017** -- *Nutrient Reference Values for Australia and New
  Zealand, Executive Summary* (`Nutrient-reference-aus-nz-executive-summary (1).pdf`)
  -- Energy, Protein, Total fat, Carbohydrate, Dietary fibre, Sodium.
- **WHO 2015** -- *Guideline: Sugars intake for adults and children*
  (`sugar_0.pdf`) -- the two free-sugars %energy thresholds.

Not included in this table (different data shape -- see the note at the
end of this notebook): **WHO NPPM 2022** (`WHO-EURO-...pdf`) gives
per-product-category composition thresholds (13 product subcategories x
their own sodium/sugar/protein/fat limits), not per-age-band nutrient
values, so it doesn't fit this table's schema and needs a separate table.

Run this notebook with its own folder (`reference_value/`) as the working
directory.

## Build reference_values.db

In [1]:
"""
Builds reference_values.db from the NHMRC and WHO source PDFs already
reviewed for this project (see reference_value/ folder). All figures below
were transcribed directly from pdftotext -layout extractions of the
official source tables and cross-checked against the table structure --
nothing here is estimated or interpolated beyond what the sources
literally state.

Schema (long/tidy format -- one row per distinct reference figure):
  age_band     '0-5m' | '6-11m' | '1y' | '2y' | '3y' | '4y'
  age_months   exact month within the band, where the source is
               month-granular (Energy Table 1 only); NULL otherwise
  nutrient     'energy' | 'protein' | 'total_fat' | 'saturated_fat' |
               'carbohydrate' | 'dietary_fibre' | 'sodium' | 'sugars'
  sex          'boys' | 'girls' | NULL (NHMRC's child macronutrient/
               mineral tables are unisex; only the Energy tables split
               by sex)
  pal_level    physical activity level label, only for Energy at 3y/4y
               (NHMRC gives 6 PAL tiers, not one number -- see note below)
  pal_value    the numeric PAL (1.2-2.2), paired with pal_level
  value_type   'EER' | 'AI' | 'EAR' | 'RDI' | 'UL' |
               'WHO_sugars_strong_10pct' | 'WHO_sugars_conditional_5pct' |
               'no_value_established'
  value_min    lower bound (equals value_max for a single-point figure)
  value_max    upper bound
  unit         'kJ/day' | 'g/day' | 'mg/day' | '%energy' -- Energy is always kJ/day
               (NHMRC Table 2's native MJ/day is converted x1000, exact, so every
               Energy row shares one unit)
  basis        'per_day_total' | 'percent_energy'
  source       which document + table the figure is from
  source_note  caveats: merged age ranges, shared life-stage values,
               explicit "no value" confirmations, etc.

Two design decisions were confirmed with the user before building (both
follow the project's standing rule: never simplify away real structure
in the official standards just to fit a tidier table):

1. Sex: NHMRC's Energy tables (Table 1, Table 2) give different EER
   figures for boys and girls at every single age point. Whether Epic 1's
   child profile will capture sex was not yet decided by the team, so
   BOTH sexes are stored as separate rows for every Energy figure -- no
   value is dropped or averaged.

2. PAL (Physical Activity Level): NHMRC Table 2 gives ages 3 and 4 as six
   different EER figures per sex (PAL 1.2 "bed rest" through PAL 2.2
   "vigorous"), not one number. No team default PAL was set yet, so all
   six tiers are stored as separate rows rather than guessing one.

A third decision, made without needing to ask (it follows directly from
the data, not a judgment call): NHMRC Table 1 (infant Energy) is
month-granular from month 1 to month 24, with extra data points at
months 15/18/21 that fall inside the "1 year" life-stage year
(12-23 months). Those points are kept too, tagged with their real
age_months, rather than discarded to keep the table tidy.

Nutrients confirmed to have NO NHMRC reference value at a given age are
stored explicitly as a row with value_min/value_max = NULL and
value_type = 'no_value_established', rather than a silently missing row
-- this keeps the "we checked and there is genuinely no reference"
finding queryable, consistent with the project's rule to never fabricate
a value and to always be explicit about coverage gaps.
"""
import sqlite3, tempfile, shutil, os
from pathlib import Path

BASE_DIR = Path.cwd()
OUT_DB = BASE_DIR / "reference_values.db"

rows = []

def add(age_band, nutrient, value_type, value_min, value_max, unit, basis,
        source, source_note=None, sex=None, pal_level=None, pal_value=None,
        age_months=None):
    rows.append(dict(age_band=age_band, age_months=age_months, nutrient=nutrient,
                      sex=sex, pal_level=pal_level, pal_value=pal_value,
                      value_type=value_type, value_min=value_min, value_max=value_max,
                      unit=unit, basis=basis, source=source, source_note=source_note))

NHMRC = "NHMRC 2017 -- Nutrient Reference Values for Australia and New Zealand, Executive Summary"

# ---- ENERGY: NHMRC Table 1 (Infants and young children, EER by month, kJ/day) ----
TABLE1 = {
    1: ("0-5m", 2000, 1800), 2: ("0-5m", 2400, 2100), 3: ("0-5m", 2400, 2200),
    4: ("0-5m", 2400, 2200), 5: ("0-5m", 2500, 2300),
    6: ("6-11m", 2700, 2500), 7: ("6-11m", 2800, 2500), 8: ("6-11m", 3000, 2700),
    9: ("6-11m", 3100, 2800), 10: ("6-11m", 3300, 3000), 11: ("6-11m", 3400, 3100),
    12: ("1y", 3500, 3200), 15: ("1y", 3800, 3500), 18: ("1y", 4000, 3800),
    21: ("1y", 4200, 4000),
    24: ("2y", 4400, 4200),
}
for month, (band, boys, girls) in TABLE1.items():
    note = "NHMRC Table 1 is month-granular; this age_band groups its monthly EER points."
    if month in (15, 18, 21):
        note += " This point (month %d) falls within the 12-23 month '1 year' life-stage span." % month
    add(band, "energy", "EER", boys, boys, "kJ/day", "per_day_total", NHMRC + ", Table 1",
        source_note=note, sex="boys", age_months=month)
    add(band, "energy", "EER", girls, girls, "kJ/day", "per_day_total", NHMRC + ", Table 1",
        source_note=note, sex="girls", age_months=month)

# ---- ENERGY: NHMRC Table 2 (Children/adolescents, EER by year x PAL, MJ/day) ----
PAL_LABELS = {1.2: "bed_rest", 1.4: "very_sedentary", 1.6: "light",
              1.8: "moderate", 2.0: "heavy", 2.2: "vigorous"}
TABLE2 = {
    3: {"boys": {1.2: 4.2, 1.4: 4.9, 1.6: 5.6, 1.8: 6.3, 2.0: 6.9, 2.2: 7.6},
        "girls": {1.2: 3.9, 1.4: 4.5, 1.6: 5.3, 1.8: 5.8, 2.0: 6.4, 2.2: 7.1}},
    4: {"boys": {1.2: 4.4, 1.4: 5.2, 1.6: 5.9, 1.8: 6.6, 2.0: 7.3, 2.2: 8.1},
        "girls": {1.2: 4.1, 1.4: 4.8, 1.6: 5.5, 1.8: 6.1, 2.0: 6.8, 2.2: 7.5}},
}
for age_yr, by_sex in TABLE2.items():
    band = "%dy" % age_yr
    for sex, pal_vals in by_sex.items():
        for pal, mj in pal_vals.items():
            kj = round(mj * 1000, 1)  # NHMRC Table 2 gives MJ/day; converted to kJ/day so
                                       # every Energy row in this table shares one unit
                                       # (Table 1 is natively kJ/day, and products.db's
                                       # energy fields are kcal/kJ, not MJ). 1 MJ = 1000 kJ
                                       # exactly -- an exact unit conversion, not an estimate.
            add(band, "energy", "EER", kj, kj, "kJ/day", "per_day_total", NHMRC + ", Table 2",
                source_note="NHMRC Table 2 states this as %s MJ/day; converted to kJ/day (x1000, exact) "
                             "so every Energy row uses one unit. 6 PAL tiers given, not a single "
                             "default; app/team must pick which to display." % mj,
                sex=sex, pal_level=PAL_LABELS[pal], pal_value=pal)

# ---- PROTEIN ----
add("0-5m", "protein", "AI", 10, 10, "g/day", "per_day_total", NHMRC + ", Table 10")
add("6-11m", "protein", "AI", 14, 14, "g/day", "per_day_total", NHMRC + ", Table 11")
for band in ("1y", "2y", "3y"):
    add(band, "protein", "EAR", 12, 12, "g/day", "per_day_total", NHMRC + ", Table 12",
        source_note="NHMRC Table 12 covers the combined 1-3 years life stage as one value, not per single year.")
    add(band, "protein", "RDI", 14, 14, "g/day", "per_day_total", NHMRC + ", Table 12",
        source_note="NHMRC Table 12 covers the combined 1-3 years life stage as one value, not per single year.")
add("4y", "protein", "EAR", 16, 16, "g/day", "per_day_total", NHMRC + ", Table 13",
    source_note="NHMRC Table 13 covers the combined 4-8 years life stage; this is not an age-4-specific figure.")
add("4y", "protein", "RDI", 20, 20, "g/day", "per_day_total", NHMRC + ", Table 13",
    source_note="NHMRC Table 13 covers the combined 4-8 years life stage; this is not an age-4-specific figure.")

# ---- TOTAL FAT ----
add("0-5m", "total_fat", "AI", 31, 31, "g/day", "per_day_total", NHMRC + ", Table 10")
add("6-11m", "total_fat", "AI", 30, 30, "g/day", "per_day_total", NHMRC + ", Table 11")
for band, tbl in (("1y", "Table 12"), ("2y", "Table 12"), ("3y", "Table 12"), ("4y", "Table 13")):
    add(band, "total_fat", "no_value_established", None, None, "g/day", "per_day_total",
        NHMRC + ", " + tbl,
        source_note="NHMRC explicitly lists no AI/EAR/RDI for total fat at this age (row shown as '-'); "
                     "only essentiality of fat type, not amount, is addressed.")

# ---- SATURATED FAT (confirmed zero coverage at every age, all 3 sources) ----
for band in ("0-5m", "6-11m", "1y", "2y", "3y", "4y"):
    add(band, "saturated_fat", "no_value_established", None, None, "g/day", "per_day_total",
        "NHMRC / WHO NPPM / WHO Sugars (all checked)",
        source_note="No source used in this project sets a saturated fat reference value below 14 years; "
                     "NHMRC's only saturated fat guidance (AMDR/SDT, Tables 23-24) is explicitly headed "
                     "'applicable to adolescents over 14yrs and adults'.")

# ---- CARBOHYDRATE ----
add("0-5m", "carbohydrate", "AI", 60, 60, "g/day", "per_day_total", NHMRC + ", Table 10")
add("6-11m", "carbohydrate", "AI", 95, 95, "g/day", "per_day_total", NHMRC + ", Table 11")
for band, tbl in (("1y", "Table 12"), ("2y", "Table 12"), ("3y", "Table 12"), ("4y", "Table 13")):
    add(band, "carbohydrate", "no_value_established", None, None, "g/day", "per_day_total",
        NHMRC + ", " + tbl,
        source_note="NHMRC explicitly lists no AI/EAR/RDI for carbohydrate at this age (row shown as '-'); "
                     "stated reason: 'limited data re essentiality on which to set EAR/RDI or AI'.")

# ---- DIETARY FIBRE ----
add("0-5m", "dietary_fibre", "no_value_established", None, None, "g/day", "per_day_total",
    NHMRC + ", Table 10", source_note="NHMRC Table 10 lists dietary fibre as 'None set' for 0-6 months.")
add("6-11m", "dietary_fibre", "no_value_established", None, None, "g/day", "per_day_total",
    NHMRC + ", Table 11", source_note="NHMRC Table 11 lists dietary fibre as 'None set' for 7-12 months.")
for band in ("1y", "2y", "3y"):
    add(band, "dietary_fibre", "AI", 14, 14, "g/day", "per_day_total", NHMRC + ", Table 12",
        source_note="NHMRC Table 12 covers the combined 1-3 years life stage as one value, not per single year.")
add("4y", "dietary_fibre", "AI", 18, 18, "g/day", "per_day_total", NHMRC + ", Table 13",
    source_note="NHMRC Table 13 covers the combined 4-8 years life stage; this is not an age-4-specific figure.")

# ---- SODIUM: AI ----
add("0-5m", "sodium", "AI", 120, 120, "mg/day", "per_day_total", NHMRC + ", Table 10")
add("6-11m", "sodium", "AI", 170, 170, "mg/day", "per_day_total", NHMRC + ", Table 11")
for band in ("1y", "2y", "3y"):
    add(band, "sodium", "AI", 200, 400, "mg/day", "per_day_total", NHMRC + ", Table 12",
        source_note="NHMRC gives this as a 200-400mg range (not a single point), covering the combined "
                     "1-3 years life stage as one value.")
add("4y", "sodium", "AI", 300, 600, "mg/day", "per_day_total", NHMRC + ", Table 13",
    source_note="NHMRC gives this as a 300-600mg range (not a single point), covering the combined "
                 "4-8 years life stage; this is not an age-4-specific figure.")

# ---- SODIUM: UL ----
add("0-5m", "sodium", "no_value_established", None, None, "mg/day", "per_day_total", NHMRC + ", Table 22",
    source_note="NHMRC Table 22 marks sodium UL as 'NP' (not possible to establish) for infants.")
add("6-11m", "sodium", "no_value_established", None, None, "mg/day", "per_day_total", NHMRC + ", Table 22",
    source_note="NHMRC Table 22 marks sodium UL as 'NP' (not possible to establish) for infants.")
for band in ("1y", "2y", "3y"):
    add(band, "sodium", "UL", 1000, 1000, "mg/day", "per_day_total", NHMRC + ", Table 22",
        source_note="NHMRC Table 22 gives this UL for the combined 1-3 years group.")
add("4y", "sodium", "UL", 1400, 1400, "mg/day", "per_day_total", NHMRC + ", Table 22",
    source_note="NHMRC Table 22 gives this UL for the combined 4-8 years group; this is not age-4-specific.")

# ---- SUGARS: WHO 2015 free-sugars guideline ----
# Stored as the %energy thresholds themselves (as WHO actually states them), not
# pre-converted to g/day -- converting requires picking one Energy value first
# (which sex? which PAL, for 3-4y?), and that choice belongs at query/display
# time in the app, not baked into the reference table.
WHO_SUGARS = "WHO 2015 -- Guideline: Sugars intake for adults and children"
for band in ("0-5m", "6-11m", "1y", "2y", "3y", "4y"):
    add(band, "sugars", "WHO_sugars_strong_10pct", 10, 10, "%energy", "percent_energy", WHO_SUGARS,
        source_note="Strong recommendation: free sugars <10% of total energy intake. WHO states this "
                     "applies 'throughout the lifecourse' with no explicit lower age boundary given.")
    add(band, "sugars", "WHO_sugars_conditional_5pct", 5, 5, "%energy", "percent_energy", WHO_SUGARS,
        source_note="Conditional recommendation (for additional health benefit): free sugars <5% of total "
                     "energy intake. Same lifecourse scope note as the 10% strong recommendation.")

print(f"Prepared {len(rows)} reference_values rows.")

if os.path.exists(OUT_DB) if False else False:
    pass

local_tmp_db = Path(tempfile.mkdtemp()) / "reference_values.db"
conn = sqlite3.connect(local_tmp_db)
cur = conn.cursor()
cur.execute("""
CREATE TABLE reference_values (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    age_band TEXT NOT NULL,
    age_months INTEGER,
    nutrient TEXT NOT NULL,
    sex TEXT,
    pal_level TEXT,
    pal_value REAL,
    value_type TEXT NOT NULL,
    value_min REAL,
    value_max REAL,
    unit TEXT NOT NULL,
    basis TEXT NOT NULL,
    source TEXT NOT NULL,
    source_note TEXT
)
""")
cols = list(rows[0].keys())
placeholders = ",".join(["?"] * len(cols))
cur.executemany(
    f"INSERT INTO reference_values ({','.join(cols)}) VALUES ({placeholders})",
    [tuple(r[c] for c in cols) for r in rows]
)
conn.commit()

n = cur.execute("SELECT count(*) FROM reference_values").fetchone()[0]
print(f"reference_values.db built: {n} rows.")
print()
print("Rows per nutrient:")
for nutrient, cnt in cur.execute("SELECT nutrient, count(*) FROM reference_values GROUP BY nutrient ORDER BY nutrient"):
    print(f"  {nutrient:<15}{cnt}")
print()
print("Rows with NO value established (explicit, honest gaps):")
for nutrient, band, cnt in cur.execute(
        "SELECT nutrient, age_band, count(*) FROM reference_values "
        "WHERE value_type='no_value_established' GROUP BY nutrient, age_band ORDER BY nutrient, age_band"):
    print(f"  {nutrient:<15}{band:<8}{cnt}")
conn.close()

shutil.copy(local_tmp_db, OUT_DB)
print()
print("Copied to:", OUT_DB)


Prepared 114 reference_values rows.
reference_values.db built: 114 rows.

Rows per nutrient:
  carbohydrate   6
  dietary_fibre  6
  energy         56
  protein        10
  saturated_fat  6
  sodium         12
  sugars         12
  total_fat      6

Rows with NO value established (explicit, honest gaps):
  carbohydrate   1y      1
  carbohydrate   2y      1
  carbohydrate   3y      1
  carbohydrate   4y      1
  dietary_fibre  0-5m    1
  dietary_fibre  6-11m   1
  saturated_fat  0-5m    1
  saturated_fat  1y      1
  saturated_fat  2y      1
  saturated_fat  3y      1
  saturated_fat  4y      1
  saturated_fat  6-11m   1
  sodium         0-5m    1
  sodium         6-11m   1
  total_fat      1y      1
  total_fat      2y      1
  total_fat      3y      1
  total_fat      4y      1

Copied to: /sessions/rcw-01uqfppw7p4f5sr8tj5ml8is/mnt/Desktop/5120 IE/FOODTRUST DATA/reference_value/reference_values.db


## Schema

| column | notes |
|---|---|
| age_band | `0-5m`, `6-11m`, `1y`, `2y`, `3y`, `4y` |
| age_months | exact month, only for Energy's month-granular NHMRC Table 1 rows |
| nutrient | energy / protein / total_fat / saturated_fat / carbohydrate / dietary_fibre / sodium / sugars |
| sex | `boys` / `girls` / NULL -- only Energy is sex-specific in NHMRC's tables |
| pal_level, pal_value | Physical Activity Level tier -- only for Energy at 3y/4y (NHMRC gives 6 tiers, not one default) |
| value_type | AI / EAR / RDI / UL / EER / WHO_sugars_strong_10pct / WHO_sugars_conditional_5pct / `no_value_established` |
| value_min, value_max | equal for a single-point figure; genuinely different for NHMRC's sodium AI ranges |
| unit, basis | e.g. `mg/day` + `per_day_total`, or `%energy` + `percent_energy`. Energy is always `kJ/day` -- NHMRC Table 2 natively states ages 3y/4y in MJ/day, converted x1000 (exact) so Energy uses one consistent unit across all 6 age bands, matching `products.db`'s kcal/kJ energy fields. |
| source, source_note | exact document+table, plus every caveat (merged age ranges, "no value exists" confirmations, etc.) |

## Two decisions confirmed with the user before building
1. **Sex** -- Epic 1's child-profile design hasn't decided whether it captures sex yet, so every
   Energy figure is stored for both boys and girls rather than guessing or averaging one.
2. **PAL default** -- the team has not chosen a default Physical Activity Level, so all 6 NHMRC
   PAL tiers are stored for ages 3y/4y Energy rather than picking one.

## Known, now-queryable gaps
`saturated_fat` has **zero** reference values at every age (checked NHMRC + WHO NPPM + WHO Sugars).
`total_fat` and `carbohydrate` have values only for 0-5m/6-11m (infant AIs); no NHMRC value exists
for 1y-4y. These are stored as explicit `no_value_established` rows, not silently missing ones.